# Семинар №3: Развёртывание нейронных сетей на собственном компьютере и в облаке


## Цели семинара

- Понять преимущества и ограничения локального / облачного запуска нейросетей
- Научиться разворачивать три типа моделей полностью оффлайн или в облаке:
  1. Большие языковые модели (LLM)
  2. Модели генерации изображений
  3. Модели генерации речи (TTS)
- Освоить разницу CPU ↔ GPU
- Научиться использовать Google Colab (и кратко другие облака) при отсутствии мощной видеокарты
- Настроить удалённый доступ к сервисам

## Почему локальное / облачное развёртывание важно в 2026 году

1. Приватность — данные не уходят на чужие серверы  
2. Нет лимитов по токенам / запросам / темам  
3. Оффлайн-работа (после загрузки моделей)  
4. Эксперименты, fine-tuning, LoRA без оплаты API  
5. Экономия (после покупки железа или при использовании бесплатного Colab)

## Обзор инструментов

| Тип модели              | Инструмент                     | Почему выбран                                 | Альтернативы (2026)                     |
|-------------------------|--------------------------------|-----------------------------------------------|------------------------------------------|
| LLM                     | Ollama + Open WebUI            | Самый простой старт, огромный каталог моделей | LM Studio, Jan.ai, GPT4All, LocalAI     |
| Генерация изображений   | stable-diffusion-webui (A1111) / Forge | Максимальная экосистема, Flux / SDXL / SD3   | ComfyUI, InvokeAI, Fooocus              |
| Генерация речи (TTS)    | Kokoro-82M + Gradio            | 82 млн параметров, мультиязычная, почти realtime | XTTSv2, Piper, MeloTTS, StyleTTS 2     |

## Аппаратные требования и скорость

| Компонент          | Минимум                     | Комфортно               | Скорость (пример)                              |
|--------------------|-----------------------------|-------------------------|------------------------------------------------|
| RAM                | 16 ГБ                       | 32 ГБ+                  | —                                              |
| Диск               | SSD ≥ 100 ГБ свободно       | SSD ≥ 300–500 ГБ        | —                                              |
| GPU (желательно)   | NVIDIA ≥ 6 ГБ VRAM          | RTX 3060–5090 (8–24 ГБ) | LLM 40–150 tok/s, Image 3–20 сек, TTS realtime |
| CPU fallback       | 8+ ядер                     | 12–16 ядер              | LLM 5–25 tok/s, Image 1–10 мин                 |

**Colab бесплатный:** T4 (16 ГБ VRAM) → LLM до ~13–14B, Flux.1-schnell ~10–30 сек/изображение

## 1. Генерация текста — Ollama / WebUI


Данное руководство описывает развёртывание локальной LLM-инфраструктуры на базе **Ollama** и веб-интерфейса **Open WebUI**.  
Рассматриваются два варианта ОС:

- Windows 10/11  
- Linux (Ubuntu/Debian и совместимые дистрибутивы)

---

### 1. Общая архитектура

```
[Browser]
     ↓
[Open WebUI]
     ↓
[Ollama API :11434]
     ↓
[LLM Model (локально)]
```

- Ollama управляет моделями  
- Open WebUI предоставляет web-интерфейс  
- Всё работает локально без внешних API  

---

### 2. Минимальные требования

#### Аппаратные

- CPU x64 с AVX2  
- RAM:
  - 8 ГБ — минимум  
  - 16–32 ГБ — рекомендуется для моделей 7B+  
- Диск: 20+ ГБ  
- GPU (опционально): NVIDIA с CUDA  

#### Программные

- Docker (рекомендуется для WebUI)  
- Либо Python 3.10+  

---

### ЧАСТЬ I — Установка на Windows

---

### 1. Установка Ollama (Windows)

#### Шаг 1. Скачивание

1. Перейдите на:  
   https://ollama.com  
2. Скачайте установщик для Windows  
3. Установите программу  

#### Шаг 2. Проверка

В командной строке Windows введите:

```powershell
ollama --version
```

---

### 2. Загрузка модели

```powershell
ollama pull llama3
```

Другие модели:

```powershell
ollama pull mistral
ollama pull qwen
ollama pull phi3
```
и т.п.

Проверка запуска:

```powershell
ollama run llama3
```

Для выхода нажмите Ctrl+z

---
Доп материалы: https://d00m4ace.com/posts/ispolzovanie-ollama---zapusk-llm-modelej-i-nastrojka-lokalno-na-windows-10/


### 3. Установка Open WebUI через Docker (Windows)

#### Шаг 1. Установка Docker Desktop

https://www.docker.com/products/docker-desktop/

Проверка:

```powershell
docker --version
```

#### Шаг 2. Запуск WebUI
Запустите Docker Desktop и обновите, если это необходимо, а затем выполните в командой строке Windows:

```powershell
docker run -d ^
  -p 3000:8080 ^
  -e OLLAMA_BASE_URL=http://host.docker.internal:11434 ^
  --name open-webui ^
  --restart always ^
  ghcr.io/open-webui/open-webui:main
```

После загрузки и настройки подождите некоторое время и откройте:

```
http://localhost:3000
```

---

### 4. Альтернатива без Docker (Windows)

```powershell
git clone https://github.com/open-webui/open-webui.git
cd open-webui
pip install -r requirements.txt
set OLLAMA_BASE_URL=http://localhost:11434
python main.py
```

---

### ЧАСТЬ II — Установка на Linux (Ubuntu/Debian)

---

### 1. Установка Ollama (Linux)

#### Шаг 1. Установка

```bash
curl -fsSL https://ollama.com/install.sh | sh
```

Проверка:

```bash
ollama --version
```

#### Шаг 2. Запуск сервиса

```bash
ollama serve
```

API по умолчанию:

```
http://localhost:11434
```

#### Шаг 3. Автозапуск

```bash
sudo systemctl enable ollama
sudo systemctl start ollama
```

---

### 2. Загрузка модели (Linux)

```bash
ollama pull llama3
ollama run llama3
```

---

### 3. Установка Open WebUI через Docker (Linux)

#### Установка Docker (Ubuntu)

```bash
sudo apt update
sudo apt install docker.io -y
sudo systemctl enable docker
sudo systemctl start docker
```

Проверка:

```bash
docker --version
```

#### Запуск WebUI

```bash
docker run -d \
  -p 3000:8080 \
  -e OLLAMA_BASE_URL=http://host.docker.internal:11434 \
  --name open-webui \
  --restart always \
  ghcr.io/open-webui/open-webui:main
```

Если `host.docker.internal` не работает:

```
http://localhost:11434
```

---

### 4. Альтернатива без Docker (Linux)

```bash
git clone https://github.com/open-webui/open-webui.git
cd open-webui
pip install -r requirements.txt
export OLLAMA_BASE_URL=http://localhost:11434
python main.py
```

---

### 5. Настройка GPU (Linux, NVIDIA)

Проверка:

```bash
nvidia-smi
```

Docker с GPU:

```bash
docker run -d \
  --gpus all \
  -p 3000:8080 \
  -e OLLAMA_BASE_URL=http://host.docker.internal:11434 \
  --name open-webui \
  ghcr.io/open-webui/open-webui:main
```

---

### 6. Проверка API

```bash
curl http://localhost:11434/api/tags
```

---

### 7. Управление моделями

```bash
ollama list
ollama rm llama3
ollama pull llama3
```

---

### 8. Типовые проблемы

#### WebUI не видит Ollama

Linux:

```bash
sudo lsof -i :11434
```

Windows:

```powershell
netstat -ano | findstr :11434
```

#### Недостаточно памяти

- 7B модель — 8–12 ГБ RAM  
- 13B модель — 16–24 ГБ RAM  

---

### 9. Рекомендованные модели

| RAM | Модель |
|------|--------|
| 8 ГБ | phi3, tinyllama |
| 16 ГБ | llama3:8b, mistral |
| 32+ ГБ | llama3:70b (с GPU) |

---


## 2. Генерация изображений — Automatic1111 / Forge

### Инструкция по установке Stable Diffusion (AUTOMATIC1111) на Windows и Linux

Данное руководство описывает развёртывание локальной системы генерации изображений на базе **Stable Diffusion WebUI (AUTOMATIC1111)**.

Поддерживаемые ОС:

- Windows 10/11  
- Linux (Ubuntu/Debian и совместимые дистрибутивы)

---

### 1. Общая архитектура

```
[Browser]
     ↓
[Stable Diffusion WebUI :7860]
     ↓
[PyTorch + CUDA / CPU]
     ↓
[Stable Diffusion Model (.ckpt / .safetensors)]
```

---

### 2. Минимальные требования

#### Аппаратные

**GPU (рекомендуется):**
- NVIDIA GPU
- 6 ГБ VRAM — минимум
- 8–12 ГБ VRAM — комфортная работа

**CPU-режим (не рекомендуется):**
- 16+ ГБ RAM
- Очень медленная генерация

#### Программные

- Python 3.10.x (не 3.11/3.12)
- Git
- CUDA (для GPU-режима)
- Драйвер NVIDIA актуальной версии

---

## ЧАСТЬ I — Установка на Windows

---

### 3. Подготовка системы (Windows)

#### Шаг 1. Установка Python 3.10 версии

Скачать:
https://www.python.org/downloads/release/python-31011/

Важно:
- Отметить "Add Python to PATH"
- Использовать версию 3.10.x
- В пути не должно быть Русских символов

Проверка:

```powershell
python --version
```

---

#### Шаг 2. Установка Git

https://git-scm.com/download/win

Проверка:

```powershell
git --version
```

---

#### Шаг 3. Проверка GPU

```powershell
nvidia-smi
```

Если команда работает — драйвер установлен корректно.


---

### 4. Клонирование AUTOMATIC1111

```powershell
git clone https://github.com/AUTOMATIC1111/stable-diffusion-webui.git
cd stable-diffusion-webui
```

---

### 5. Первый запуск

```powershell
webui-user.bat
```

При первом запуске:

- Скачивается PyTorch
- Устанавливаются зависимости
- Запускается сервер

Открыть в браузере:

```
http://localhost:7860
```

---

### 6. Установка модели

Модели размещаются в:

```
stable-diffusion-webui/models/Stable-diffusion/
```

Поддерживаемые форматы:

- `.ckpt`
- `.safetensors` (рекомендуется)

Популярные модели:

- SD 1.5
- SDXL

После добавления модели — выбрать её в интерфейсе.

---

### 7. Параметры запуска (Windows)

Файл:

```
webui-user.bat
```

Можно добавить аргументы:

```bat
set COMMANDLINE_ARGS=--xformers --medvram --api
```

Полезные параметры:

- `--xformers` — ускорение
- `--medvram` — для 6–8 ГБ VRAM
- `--lowvram` — для 4–6 ГБ VRAM
- `--api` — включить API
- `--listen` — доступ из сети

---

## ЧАСТЬ II — Установка на Linux (Ubuntu)

---

### 8. Установка зависимостей

```bash
sudo apt update
sudo apt install git python3.10 python3.10-venv -y
```

---

### 9. Проверка GPU

```bash
nvidia-smi
```

Если CUDA не установлена — установить драйвер NVIDIA.

---

### 10. Клонирование репозитория

```bash
git clone https://github.com/AUTOMATIC1111/stable-diffusion-webui.git
cd stable-diffusion-webui
```

---

### 11. Запуск

```bash
./webui.sh
```

Открыть:

```
http://localhost:7860
```

---

### 12. Параметры запуска (Linux)

```bash
./webui.sh --xformers --medvram --api
```

---

### 13. Использование CPU (если нет GPU)

Windows:

```bat
set COMMANDLINE_ARGS=--use-cpu all
```

Linux:

```bash
./webui.sh --use-cpu all
```

⚠ Генерация будет очень медленной.

---

### 14. Оптимизация для малой VRAM

Для 6–8 ГБ VRAM:

```
--medvram --xformers
```

Для 4–6 ГБ:

```
--lowvram --xformers
```

Для SDXL рекомендуется 12+ ГБ VRAM.

---

### 15. Проверка работы API

Если включен `--api`:

```
http://localhost:7860/docs
```

Пример запроса:

```bash
curl http://localhost:7860/sdapi/v1/txt2img
```

---

### 16. Обновление WebUI

```bash
git pull
```

Windows — внутри папки проекта:

```powershell
git pull
```

---

### 17. Типовые проблемы

#### CUDA not available

- Проверить `nvidia-smi`
- Обновить драйвер
- Убедиться, что PyTorch установлен с CUDA

#### RuntimeError: CUDA out of memory

- Использовать `--medvram` или `--lowvram`
- Уменьшить resolution
- Уменьшить batch size

#### Python version error

Использовать строго Python 3.10.x

---

### 18. Рекомендуемые разрешения

| VRAM | Разрешение |
|------|------------|
| 6 ГБ | 512x512 |
| 8 ГБ | 768x768 |
| 12+ ГБ | 1024x1024 |

---

### 19. Структура проекта

```
stable-diffusion-webui/
 ├── models/
 │    └── Stable-diffusion/
 ├── extensions/
 ├── embeddings/
 ├── outputs/
 └── webui-user.bat / webui.sh
```

---


### В Colab (рекомендуется Forge для Flux)

Популярные ноутбуки 2026:
- Forge + Flux: https://colab.research.google.com/github/camenduru/flux-jupyter/blob/main/flux.1-dev_jupyter.ipynb
- Классический A1111: https://colab.research.google.com/github/TheLastBen/fast-stable-diffusion/blob/main/fast_stable_diffusion_AUTOMATIC1111.ipynb

→ Runtime → T4 GPU → Run all → публичная ссылка gradio / ngrok

## 3. TTS — Kokoro-82M

### Локальная установка

### Инструкция по установке и запуску Kokoro 82M на Windows и Linux

Данное руководство описывает развёртывание локальной LLM **Kokoro 82M** с веб-интерфейсом для взаимодействия через браузер.  
Поддерживаемые ОС:

- Windows 10/11  
- Linux (Ubuntu/Debian и совместимые дистрибутивы)

---

### 1. Общая архитектура

```
[Browser]
     ↓
[WebUI (Gradio/Streamlit)]
     ↓
[Kokoro 82M LLM]
     ↓
[PyTorch + CUDA / CPU]
```

- Модель Kokoro 82M работает локально  
- WebUI обеспечивает удобный интерфейс  
- GPU ускоряет генерацию ответов  

---

### 2. Минимальные требования

#### Аппаратные

- CPU x64  
- RAM:
  - 4–8 ГБ — для базового использования  
  - 8+ ГБ — комфортная работа  
- GPU (опционально):
  - NVIDIA с CUDA для ускорения
  - 4–6 ГБ VRAM — минимально
  - 8+ ГБ VRAM — рекомендуется  

#### Программные

- Python 3.10+  
- Git  
- PyTorch 2.x с поддержкой CUDA (для GPU)  

---

## ЧАСТЬ I — Установка на Windows

---

### 3. Подготовка системы

#### Шаг 1. Установка Python

Скачать Python 3.10.x:  
https://www.python.org/downloads/release/python-31011/

Важно:  
- Отметить "Add Python to PATH"  

Проверка:

```powershell
python --version
```

#### Шаг 2. Установка Git

Скачать: https://git-scm.com/download/win

Проверка:

```powershell
git --version
```

#### Шаг 3. Проверка GPU (опционально)

```powershell
nvidia-smi
```

---

### 4. Клонирование репозитория Kokoro 82M

```powershell
git clone https://github.com/kokoroai/kokoro-82m.git
cd kokoro-82m
```

---

### 5. Установка зависимостей

```powershell
pip install -r requirements.txt
```

Для GPU:

```powershell
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
```

Для CPU:

```powershell
pip install torch torchvision torchaudio
```

---

### 6. Загрузка модели

Скачать веса Kokoro 82M и положить в папку:

```
kokoro-82m/models/
```

Поддерживаемые форматы:

- `.pt`  
- `.safetensors`

---

### 7. Запуск WebUI

Для Windows:

```powershell
python app.py
```

- По умолчанию WebUI запускается на:

```
http://localhost:7860
```

- Для изменения порта:

```powershell
python app.py --port 8080
```

---

## ЧАСТЬ II — Установка на Linux (Ubuntu/Debian)

---

### 8. Подготовка системы

```bash
sudo apt update
sudo apt install git python3.10 python3.10-venv -y
```

Проверка GPU:

```bash
nvidia-smi
```

---

### 9. Клонирование репозитория

```bash
git clone https://github.com/kokoroai/kokoro-82m.git
cd kokoro-82m
```

---

### 10. Установка зависимостей

```bash
python3 -m venv venv
source venv/bin/activate
pip install -r requirements.txt
```

GPU:

```bash
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
```

CPU:

```bash
pip install torch torchvision torchaudio
```

---

### 11. Загрузка модели

Скачать веса Kokoro 82M и положить в:

```
kokoro-82m/models/
```

---

### 12. Запуск WebUI

```bash
python app.py
```

- По умолчанию:

```
http://localhost:7860
```

- Изменение порта:

```bash
python app.py --port 8080
```

---

### 13. Настройки GPU и CPU

- Для GPU — PyTorch автоматически использует CUDA  
- Для CPU — добавить флаг:

```bash
python app.py --device cpu
```

---

### 14. Проверка работы модели

- В браузере открыть WebUI  
- Ввести текстовый запрос  
- Проверить генерацию ответа  

---

### 15. Обновление Kokoro 82M

```bash
git pull
```

- Для обновления зависимостей:

```bash
pip install -r requirements.txt --upgrade
```

---

### 16. Типовые проблемы

#### Ошибка CUDA not available

- Проверить `nvidia-smi`  
- Убедиться, что PyTorch установлен с поддержкой CUDA  
- Обновить драйвер NVIDIA

#### Недостаточно памяти

- Использовать CPU-режим  
- Ограничить длину запроса  
- Использовать меньшие модели, если доступны

#### Python version error

- Использовать строго Python 3.10.x

---

### 17. Структура проекта Kokoro 82M

```
kokoro-82m/
 ├── models/         # веса модели
 ├── app.py          # запуск WebUI
 ├── requirements.txt
 ├── utils/          # вспомогательные скрипты
 └── logs/
```

---

### 18. Рекомендации по производительности

| VRAM | Режим |
|------|-------|
| 4–6 ГБ | базовые запросы, ограничение длины |
| 8+ ГБ | комфортная работа, длинные запросы |
| CPU только | небольшие тесты, медленно |

---


### Вариант запуска Kokoro 82M на Google Colab

Если нет мощного локального GPU, модель можно запускать в облаке через Google Colab.

---

#### 1. Создание нового ноутбука

1. Перейдите на [https://colab.research.google.com](https://colab.research.google.com)  
2. Создайте новый ноутбук  
3. Выберите **Среда выполнения → Изменить тип среды → GPU**  

---

#### 2. Установка зависимостей

```python
# Клонируем репозиторий
!git clone https://github.com/kokoroai/kokoro-82m.git
%cd kokoro-82m

# Установка зависимостей
!pip install -r requirements.txt

# Установка PyTorch с поддержкой CUDA
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
```

---

#### 3. Загрузка модели

Скачайте веса Kokoro 82M и загрузите их на Colab:

```python
from google.colab import files

# Загрузка файлов модели
uploaded = files.upload()
# Переносим в папку models/
import shutil
import os

for filename in uploaded.keys():
    shutil.move(filename, os.path.join("models", filename))
```

---

#### 4. Запуск WebUI

```python
!python app.py --port 7860 --share
```

- Параметр `--share` создаёт публичный временный URL для доступа через браузер  
- По умолчанию WebUI на порту 7860  

---

#### 5. Доступ к интерфейсу

- После запуска появится ссылка вида:

```
https://xxxx.gradio.app
```

- Перейдите по ссылке и используйте WebUI для генерации текста через Kokoro 82M  

---

#### 6. Особенности работы на Colab

- Сессия ограничена по времени (~12 часов)  
- Используемый GPU — ограниченный по VRAM  
- Для больших моделей может потребоваться **swap** или уменьшение параметров запроса  

---

#### 7. Полезные команды

- Проверка GPU:

```python
!nvidia-smi
```

- Обновление репозитория:

```python
!git pull
!pip install -r requirements.txt --upgrade
```


## 4. Организация удалённого доступа к локальным нейросетям

(для Ollama, Open WebUI, Automatic1111 / Forge, Kokoro TTS и других сервисов на портах 11434, 3000, 7860, 7861 и т.д.)

### Зачем нужен удалённый доступ

- Показать работу нейросети с телефона / другого компьютера
- Тестирование с разных устройств в одной сети или через интернет

### Вариант 1 — Самый простой и быстрый: Cloudflare Tunnel (рекомендуется для семинара)

**Преимущества**  
- Полностью бесплатно  
- Не требует регистрации и токенов  
- Автоматический HTTPS  
- Очень быстро запускается (5–10 секунд)

**Недостатки**  
- Случайный адрес каждый раз (меняется при перезапуске)  
- Иногда может быть очередь на бесплатных туннелях

**Установка cloudflared (один раз)**

**Windows**  
1. Перейти → https://developers.cloudflare.com/cloudflare-one/connections/connect-apps/install-and-setup/tunnel-guide/local/  
2. Скачать cloudflared-windows-amd64.exe  
3. Переименовать в `cloudflared.exe` и положить в удобную папку (например, C:\Tools)

**Linux**  
```bash
wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
chmod +x cloudflared
sudo mv cloudflared /usr/local/bin/
```

### Запуск тоннеля для различных сервисов:

```bash
# Для Automatic1111 / Forge (порт 7860)
cloudflared tunnel --url http://localhost:7860

# Для Open WebUI (порт 3000)
cloudflared tunnel --url http://localhost:3000

# Для Ollama API (порт 11434)
cloudflared tunnel --url http://localhost:11434

# Для Gradio/Kokoro TTS (порт 7861)
cloudflared tunnel --url http://localhost:7861
```

**После запуска вы увидете строку вида:**

Your tunnel is available at: https://random-words.trycloudflare.com

### Вариант 2 — ngrok (с бесплатным статическим доменом)

**Преимущества**  
- Можно получить постоянный адрес вида `https://ваш-домен.ngrok-free.app` (после регистрации)  
- Удобный веб-дашборд с историей запросов и статистикой  
- Поддержка базовой аутентификации и ограничений трафика  
- Работает стабильно даже при перезапуске туннеля (с сохранённым доменом)

**Недостатки**  
- Требуется бесплатная регистрация  
- Бесплатный тариф имеет лимиты:  
  • ~1 ГБ трафика в месяц  
  • 1 одновременное соединение  
  • случайный адрес без регистрации  
- Иногда требуется подтверждение по email/капча при первом запуске

#### Шаг 1. Регистрация и получение authtoken

1. Перейдите на сайт: https://ngrok.com  
2. Нажмите **Sign up** (регистрация через GitHub / Google / email — бесплатно)  
3. После входа зайдите в дашборд → **Your Authtoken**  
4. Скопируйте длинную строку вида:  2gabcdef1234567890ijklmnopqrstuvwxyz_ВАШ_ТОКЕН


#### Шаг 2. Установка ngrok

**Windows**  
1. Перейдите → https://ngrok.com/download  
2. Скачайте **ngrok.exe** (Windows ZIP)  
3. Распакуйте и положите `ngrok.exe` в удобную папку (например, `C:\Tools\ngrok`)  
4. (Опционально) Добавьте эту папку в системную переменную PATH

**Linux**  
```bash
# Самый простой и рекомендуемый способ
sudo snap install ngrok

# Или через официальный репозиторий
curl -s https://ngrok-agent.s3.amazonaws.com/ngrok.asc | \
sudo tee /etc/apt/trusted.gpg.d/ngrok.asc >/dev/null
echo "deb https://ngrok-agent.s3.amazonaws.com buster main" | \
sudo tee /etc/apt/sources.list.d/ngrok.list
sudo apt update && sudo apt install ngrok
```

#### Шаг 3. Авторизация
Откройте терминал / PowerShell / cmd и выполните:
```bash
ngrok config add-authtoken ВАШ_ТОКЕН_ИЗ_ДАШБОРДА
```

#### Шаг 4. Запуск туннеля

После регистрации в дашборде ngrok → Cloud Edge → Domains → создайте бесплатный домен (например my-seminar-2026.ngrok-free.app)
Затем запускайте туннель с указанием этого домена:

```bash
# Для Automatic1111 / Forge (порт 7860)
ngrok http --domain=my-seminar-2026.ngrok-free.app 7860

# Для Open WebUI (порт 3000)
ngrok http --domain=my-seminar-2026.ngrok-free.app 3000

# Для Ollama API (порт 11434)
ngrok http --domain=my-seminar-2026.ngrok-free.app 11434

# Для Gradio/Kokoro TTS (порт 7861)
ngrok http --domain=my-seminar-2026.ngrok-free.app 7861
```

Если вы не хотите / не успели создать домен — можно запустить без --domain:

```bash
ngrok http 7860
```
Тогда каждый раз будет выдаваться новый временный адрес вида
https://abcd-1234-5678-90ef.ngrok-free.app

#### Шаг 5. Защита доступа (обязательно для публичного использования!):

**Automatic1111 / Forge**
Добавьте в webui-user.bat или webui-user.sh:
```bash
set COMMANDLINE_ARGS=--gradio-auth username:strongpassword123
```
**Gradio (Kokoro TTS)**
В launch() добавьте:
```python
demo.launch(..., auth=("user", "strongpass"))
```
**Open WebUI**
- Войдите в интерфейс → Settings → Authentication → включите и задайте пароль

## Практическое задание

1. Сгенерировать короткий рассказ (LLM)
2. Составить промпт с описанием ключевой сцены → сгенерировать изображение
3. Озвучить финальный текст голосом TTS
- Настроить удалённый доступ (ngrok / --listen)
- Показать работу с телефона
- Сравнить время на CPU / GPU / Colab
- Установить хотя бы две модели локально ИЛИ в Colab
- Попробовать альтернативные модели: Flux.1-dev, Qwen 2.5 14B, XTTS-v2
- Короткий вывод: что понравилось / не понравилось / сколько места заняли модели

## Полезные ссылки

- Ollama → https://ollama.com
- A1111 → https://github.com/AUTOMATIC1111/stable-diffusion-webui
- Kokoro-82M → https://huggingface.co/hexgrad/Kokoro-82M
- Colab примеры Flux → поиск "flux colab forge 2026"

